# God of War — PAL function map (Colab, no local compute)

Maps the ~6,400 function addresses of the USA port (`SCUS_973.99`) onto the PAL
build (`SCES_531.33`) by fingerprinting the compiled MIPS routines. No Ghidra,
no decompiler, no emulator: only the two ELFs and the function lists.

**What to upload** (three small files, from your own private builder):

| file | where |
|---|---|
| `SCUS_973.99` | `gow-engine-builder/game/SCUS_973.99` |
| `SCES_531.33` | `gow-engine-builder/game/SCES_531.33` |
| `functions-usa.csv` | `gow-engine-builder/port-usa/functions-usa.csv` |

The files stay in this Colab VM; nothing is uploaded to a server.

**Result:** `pal_map.csv` with `usa_addr,usa_name,pal_addr,method,score`. Exact
matches mean the routine is byte-identical up to addresses; `fuzzy` means the
routine changed and was matched by shape (review those manually).

In [ ]:
!pip -q install pyelftools

## 1. Get the public function map and the matcher

`port/functions.csv` is the PAL Ghidra function list already published in the
port repository; `tools/analysis/elf_fn_match.py` is the matcher.

In [ ]:
!git clone --depth 1 -b main https://github.com/CodeAbhi826/God-Of-War-Native-Port.git portmap
!ls -l portmap/port/functions.csv portmap/tools/analysis/elf_fn_match.py

## 2. Upload your two ELFs and the USA function list

In [ ]:
from google.colab import files
uploaded = files.upload()
print("uploaded:", sorted(uploaded))

## 3. Match

The `--check` list is a set of addresses that the USA override file patches by
hand (heap, GS display, pad2, movies, timers). It prints their PAL equivalents
immediately, which is what the PAL override needs.

In [ ]:
CHECKS = ",".join([
    "0x00279600",  # IPU init
    "0x0028C0C8",  # rand
    "0x0014C278",  # operator new (object guard)
    "0x0014C2F0",  # operator delete
    "0x0013DC78",  # game heap alloc
    "0x00182DD8",  # GS FINISH handler
    "0x00182E88",  # GS FINISH install
    "0x00183360",  # display setup
    "0x00180E50",  # path select
    "0x00294A30",  # DelayThread
    "0x002998C0",  # InitTimer
    "0x00299A20",  # StartTimerSystemTime
    "0x0027B7E8",  # pad2 init
    "0x0027B9F0",  # pad2 read
    "0x0027BB98",  # pad2 get state
    "0x00296C48",  # sceSifInitRpc (bind)
    "0x00294990",  # iWakeupThread (bind)
    "0x0027AB00",  # sceCdReadDvdDualInfo
    "0x0026BF28",  # 989snd command
    "0x00298CE8",  # sceSifLoadStartModuleBuffer
    "0x00188B80",  # movie stream service
    "0x0018A788",  # movie video ring write
    "0x0018AAA8",  # movie no-data
    "0x00188F20",  # movie ready
    "0x0017A1D0",  # VSync wait
    "0x0017A2A0",  # DMA wait
    "0x001BA4F8",  # bonus movie
])

!python portmap/tools/analysis/elf_fn_match.py \
    --usa SCUS_973.99 \
    --pal SCES_531.33 \
    --usa-functions functions-usa.csv \
    --pal-functions portmap/port/functions.csv \
    --check "$CHECKS" \
    --out pal_map.csv

## 4. Download the map

In [ ]:
from google.colab import files
files.download("pal_map.csv")